# Qwen RSI: 50-task exploratory comparison

This notebook compares the future trained Qwen3-14B RSI adapter with its untrained base on the same 50 tasks. The 50 task IDs must be separate from the subsequent 200-task evaluation. This check is exploratory; it does not establish RSI improvement or ARC-AGI-2 competitiveness.

**Status: waiting for a trained adapter and recorded comparison.** The notebook reads an existing results file or an explicitly selected interrupted-run record; it does not train or run a model.


In [ ]:
from pathlib import Path
import json
import sys

roots = [Path.cwd(), *Path.cwd().parents]
ROOT = next((p for p in roots if (p / 'src/arc/rsi_exploratory_report.py').is_file()), None)
if ROOT is None:
    raise RuntimeError('Start this notebook from the repository or a subdirectory.')
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))
from src.arc.rsi_exploratory_report import summarize_exploratory
from src.arc.rsi_exploratory_incomplete import read_incomplete_exploratory
from IPython.display import display, Markdown
import pandas as pd

RESULTS_PATH = ROOT / 'artifacts/rsi-50/exploratory-results.json'

# For an interrupted run, set all three values explicitly.
PARTIAL_RUN_PATH = None
PARTIAL_REGISTRATION_PATH = None
PARTIAL_ADAPTER_SHA256 = None
SPLIT_PATH = ROOT / 'research/rsi/exploratory-split-proposal.json'


## Input and interpretation

The results file must contain `kind`, the pinned `model` and `revision`, `task_ids` (50), `confirmatory_task_ids` (200 disjoint IDs), and `arms` for `base` and `trained`. Each arm has an `adapter`, identical `settings`, and a `results` list. The base adapter is `none`; the trained adapter is its SHA-256. Each result has `task_id`, `task_type` (or null), `solved`, `failure_reason` (or null), `elapsed_seconds`, `prompt_tokens`, `completion_tokens`, and two `seeds`.

Task types are supplied annotations, not official ARC labels. Record the annotation method and sources alongside the results. Time and tokens should be captured per task for each arm; use null when unavailable. The summary checks consistency of supplied records, not their provenance or actual execution.

For an interrupted run, set the partial run path, its registration path and adapter digest above. The notebook reads only completed rows, reports the interruption, and keeps the stage incomplete. The supplied identities and rows remain unaudited.


In [ ]:
if PARTIAL_RUN_PATH is not None:
    import hashlib
    if PARTIAL_REGISTRATION_PATH is None or PARTIAL_ADAPTER_SHA256 is None:
        raise ValueError('An interrupted run requires its registration and adapter digest.')
    registration = json.loads(PARTIAL_REGISTRATION_PATH.read_text())
    registration_raw = json.dumps(registration, sort_keys=True, separators=(',', ':'), ensure_ascii=False,
                                  allow_nan=False).encode()
    split = json.loads(SPLIT_PATH.read_text())
    view = read_incomplete_exploratory(root=PARTIAL_RUN_PATH, registration=registration,
        registration_sha256=hashlib.sha256(registration_raw).hexdigest(),
        task_ids=split['exploratory_task_ids'], confirmatory_task_ids=split['confirmatory_task_ids'],
        candidate_adapter=PARTIAL_ADAPTER_SHA256)
    data, summary = view['data'], view['summary']
    display(Markdown('**Interrupted, unaudited comparison.** Completed rows only; interruption: ' +
                     str(view['failure_type'])))
elif RESULTS_PATH.is_file():
    if (RESULTS_PATH.parent / 'exploratory.incomplete.json').exists():
        raise ValueError('This run is incomplete; use the partial run settings above.')
    data = json.loads(RESULTS_PATH.read_text())
    summary = summarize_exploratory(data)
    display(Markdown('**Exploratory, caller-supplied results.** Independent provenance is not established.'))
else:
    data = None
    summary = None
    display(Markdown('**No recorded comparison found.** Train and audit the 14B RSI adapter and complete the required execution gates before producing results.'))


## Correct answers and measurement coverage

Counts use completed task records. Totals are shown only when all 50 tasks have a measurement; a missing value is unknown, not zero.


In [ ]:
if summary is not None:
    rows = []
    for arm, result in summary['arms'].items():
        rows.append({'arm': arm, 'evaluated': result['evaluated'], 'correct': result['correct'],
                     'elapsed_seconds_total': result['elapsed_seconds_total'], 'time_coverage': result['time_coverage'],
                     'total_tokens': result['total_tokens'], 'token_coverage': result['token_coverage']})
    display(pd.DataFrame(rows).set_index('arm'))
    display(Markdown('Complete paired records: **' + str(summary['complete']) + '**.'))


## Performance by task type and failure reasons

These counts describe supplied labels and recorded failures. A missing task type is shown as `unclassified`; a missing reason for an incorrect result is shown as `unknown`.


In [ ]:
if summary is not None:
    types = [{'arm': arm, 'task_type': kind, **counts}
             for arm, result in summary['arms'].items() for kind, counts in result['types'].items()]
    failures = [{'arm': arm, 'reason': reason, 'count': count}
                for arm, result in summary['arms'].items() for reason, count in result['failure_reasons'].items()]
    display(pd.DataFrame(types))
    display(pd.DataFrame(failures))


## Review before the full evaluation

Inspect task-level results, incomplete records, error categories, timing, token coverage, model and adapter identities, and whether the two arms used the same settings and seeds. Keep the exploratory tasks separate from the 200-task evaluation. Any choice made after reviewing these results must be documented before the later evaluation.


In [ ]:
if data is not None:
    for arm in ('base', 'trained'):
        display(Markdown('### ' + arm))
        display(pd.DataFrame(data['arms'][arm]['results']))
